In [ ]:
import json
from datetime import datetime, timezone

from dbx_tools.graphiti.runtime import GraphitiRuntime
from graphiti_core.nodes import EpisodicNode

"""Read an episode written by a separate GraphitiRuntime notebook task."""

DATABASE_URL = "projects/lfp-chat-db/branches/production/endpoints/primary"
MARKER = dbutils.jobs.taskValues.get(taskKey="write_memory", key="marker")
EPISODE_UUID = dbutils.jobs.taskValues.get(taskKey="write_memory", key="episode_uuid")


def _episode_record(episode):
    """Return JSON-safe episode fields used by the task result."""
    return {
        "uuid": str(episode.uuid),
        "name": episode.name,
        "group_id": episode.group_id,
        "content": episode.content,
        "source_description": episode.source_description,
        "source": getattr(episode.source, "value", episode.source),
        "valid_at": episode.valid_at.isoformat() if episode.valid_at else None,
        "created_at": episode.created_at.isoformat() if episode.created_at else None,
    }


async def _read_memory():
    """Verify that a fresh GraphitiRuntime can read the persisted episode."""
    runtime = GraphitiRuntime.from_options({"databaseUrl": DATABASE_URL})
    await runtime.start()
    try:
        direct = await EpisodicNode.get_by_uuid(runtime.graphiti.driver, EPISODE_UUID)
        grouped = await EpisodicNode.get_by_group_ids(
            runtime.graphiti.driver,
            ["main"],
            limit=100,
        )
        retrieved = await runtime.graphiti.retrieve_episodes(
            reference_time=datetime.now(timezone.utc),
            last_n=100,
            group_ids=["main"],
        )
        matches = [
            episode
            for episode in [direct, *grouped, *retrieved]
            if str(episode.uuid) == EPISODE_UUID or MARKER in episode.content
        ]
        if not matches:
            raise AssertionError(
                f"Fresh GraphitiRuntime did not find episode {EPISODE_UUID} with marker {MARKER}"
            )
        return {
            "marker": MARKER,
            "episode_uuid": EPISODE_UUID,
            "found": True,
            "matched_uuid": str(matches[0].uuid),
            "matched_content": matches[0].content,
            "direct": _episode_record(direct),
        }
    finally:
        await runtime.close()


result = await _read_memory()
dbutils.notebook.exit(json.dumps(result))